# C1 · What a raster is: the Kathmandu Valley as a grid of numbers

> **Time:** about 2 hours  
> **You need:** Part A (especially A8 · NumPy arrays) and Part B (especially B4 · Coordinate reference systems in practice).  
> **By the end you can:**
> 1. explain the parts of a **raster**: cells, rows and columns, cell size, the **affine transform**, the CRS, the data type, **nodata** and **bands**,
> 2. open a GeoTIFF with `rasterio`, read its metadata, and load its values into a NumPy array,
> 3. convert between rows and columns and map coordinates, find the elevation at a place, and write a small raster of your own.

All of Part B used **vector** data: points, lines and polygons. In Lesson B1 you met the other way of storing geography, the **raster**: a grid of cells, each holding one value. Elevation, rainfall, temperature and satellite images are all rasters.

Part C works with one real raster: a **digital elevation model** (DEM) of the Kathmandu Valley and the Melamchi area of Sindhupalchok. It is a clip of the Copernicus DEM, with one value about every 30 m.

*Elevation: Copernicus DEM GLO-30 (© DLR e.V. 2010-2014 and © Airbus Defence and Space GmbH 2014-2018, provided under COPERNICUS by the European Union and ESA).* The full licence notice is in `data/README.md`; read it once.

**How to practise:** after every step there are three questions. **Q1** needs one line of code, **Q2** two lines, **Q3** about three. Type your answer in the empty cell under each question, run it, then open **Show solution** to check. Try first, even if you are not sure: struggling for a few minutes is how the skill grows.

**If results look strange** (a name that is not defined, numbers that differ from the text), use **Kernel → Restart Kernel and Run All Cells**. The lessons expect every cell to run in order, from the top.

<!-- pygile-nav -->
**PyGILE Course** · lesson 25 of 39 · [Course home](../README.md)  
← Previous: [B12 · Interactive web maps: Nepal you can zoom](../B_Vector/B12_Interactive_web_maps.ipynb) · Next: [C2 · Plotting rasters: an honest map of the Kathmandu Valley DEM](C02_Plotting_rasters.ipynb) →

In [ ]:
# Optional: marks your answers as you go. Delete this cell to turn it off.
%run ../tools/tutor_on.py

## 1. A grid of cells

A **raster** stores a map as a grid of equal squares called **cells** (or **pixels**, "picture elements"). Every cell holds one value: an elevation, a rainfall total, a land-cover code.

The cells sit in **rows** and **columns**. Row 0 is the top (northern) edge, and column 0 the left (western) edge, as with the arrays of Lesson A8.

The ground width of one cell is the **cell size**, also called the **resolution**. Small cells mean fine detail and large files. A 30 m DEM is called "higher resolution" than a 90 m DEM.

![The same river stored as vector data (left) and as raster data (right)](../images/vector_vs_raster.jpg)

*Image: the pyGIS project by Michael Mann (GPL-3.0).*

In Lesson A8 you built a made-up mountain as a 5 × 6 array with 500 m cells. That array holds the **values** of a raster. It does not yet say **where** on Earth it lies.

**Predict:** how many km wide is the made-up mountain grid?

In [ ]:
import math
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
from rasterio.transform import Affine, from_origin, rowcol, xy

DATA = Path("..") / "data"
OUT = Path("..") / "outputs"
OUT.mkdir(exist_ok=True)

elev_m = np.array([
    [4210, 4350, 4480, 4520, 4400, 4290],   # row 0 (north edge)
    [4380, 4620, 4890, 4950, 4700, 4460],   # row 1
    [4510, 4870, 5230, 5410, 5020, 4630],   # row 2
    [4470, 4790, 5150, 5290, 4980, 4590],   # row 3
    [4300, 4520, 4760, 4810, 4650, 4410],   # row 4 (south edge)
])
cell_size_m = 500   # each cell covers 500 m x 500 m of ground (made-up values, as in A8)

n_rows, n_cols = elev_m.shape
print(n_rows, "rows and", n_cols, "columns")
print("width:", n_cols * cell_size_m / 1000, "km")
print("height:", n_rows * cell_size_m / 1000, "km")

The grid has 5 rows and 6 columns. Six columns of 500 m make 3.0 km from west to east; five rows make 2.5 km from north to south.

The numbers in the array are only elevations. Nothing in `elev_m` says whether it lies in Nepal or in the sea. The next step adds that information.

### Practice

**Q1 · one line:** How many cells does `elev_m` hold? Use the attribute from A8 that counts all values.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
elev_m.size
```

30 cells: 5 rows × 6 columns. Each cell holds one elevation.
</details>

**Q2 · two lines:** Imagine the same 6 columns with cells of 30 m, like the DEM in this lesson. Store `cell_30_m = 30` and print the width of the grid in km.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
cell_30_m = 30
print(n_cols * cell_30_m / 1000)
```

0.18 km, only 180 m. With smaller cells, the same number of cells covers much less ground. Finer resolution costs more cells for the same area.
</details>

**Q3 · three lines:** The DEM of this lesson has 1,620 rows and 2,160 columns. Work out how many cells it holds. Each value takes 2 bytes of memory: print the total in megabytes (1 MB = 1,000,000 bytes).

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
dem_cells = 1620 * 2160
print(dem_cells)
print(dem_cells * 2 / 1000000, "MB")
```

3,499,200 cells, which take 6.9984 MB, about 7 MB. That is small for a raster: one satellite image can hold hundreds of millions of cells.
</details>

## 2. Placing the grid: the affine transform

To place a grid on the map you need two facts: where its **top-left corner** is, and how big each cell is in x and in y.

Rasters store these facts as an **affine transform**: six numbers, called `a, b, c, d, e, f`, that turn a column and a row into map coordinates:

- x = a × col + b × row + c
- y = d × col + e × row + f

| Number | Meaning | Usual value |
| --- | --- | --- |
| `a` | cell width in x | positive |
| `e` | cell height in y | **negative** |
| `c`, `f` | x and y of the grid's top-left corner | the corner |
| `b`, `d` | rotation or shear | 0 for north-up rasters |

Why is `e` negative? Row numbers grow **downwards**, towards the south, while y (northing or latitude) grows **upwards**, towards the north. Each step down one row lowers y by one cell.

`from_origin(west, north, x_size, y_size)` builds a transform from the corner and the cell sizes. You give both sizes as positive numbers; it makes `e` negative for you.

Below, the made-up mountain is placed at a **made-up position** in UTM zone 45N (Lesson B4): its top-left corner at easting 500,000 m and northing 3,000,000 m.

In [ ]:
toy_transform = from_origin(500000, 3000000, cell_size_m, cell_size_m)
toy_transform

In [ ]:
print(toy_transform.a, toy_transform.e)   # cell width and cell height
print(toy_transform.c, toy_transform.f)   # x and y of the top-left corner
print(toy_transform @ (0, 0))             # (col 0, row 0): the top-left corner
print(toy_transform @ (6, 5))             # (col 6, row 5): the bottom-right corner

The transform shows six numbers on two lines: `a, b, c` on the first and `d, e, f` on the second. Here `a` is 500.0 and `e` is −500.0: each column moves 500 m east, each row 500 m south.

`transform @ (col, row)` applies the transform to a column and a row, in that order: **x first**, as always since Lesson B1. The `@` sign means matrix multiplication; older code writes `*` instead, which still works but is being phased out.

`(0, 0)` gives the top-left corner, (500000.0, 3000000.0). `(6, 5)` is one step past the last column and the last row, so it gives the **bottom-right corner** of the grid: (503000.0, 2997500.0). The grid covers 3,000 m × 2,500 m, as in step 1.

### Practice

**Q1 · one line:** Where is the top-left corner of the summit cell, at row 2 and column 3? Remember the order: column first.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
toy_transform @ (3, 2)
```

(501500.0, 2999000.0): 3 columns × 500 m east of the grid's corner, and 2 rows × 500 m south of it.
</details>

**Q2 · two lines:** Work out the same corner by hand with the formula, using `toy_transform.a`, `.c`, `.e` and `.f` (`b` and `d` are 0). Print x, then y.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(toy_transform.a * 3 + toy_transform.c)
print(toy_transform.e * 2 + toy_transform.f)
```

501500.0 and 2999000.0, the same as `@`. The transform is only this arithmetic, stored with the raster. Because `e` is negative, adding rows makes y smaller.
</details>

**Q3 · three lines:** The **centre** of a cell is half a cell further right and down than its corner. Print the centre of the summit cell with `toy_transform @ (3.5, 2.5)`. Then print the centre of the bottom-right cell (row 4, column 5).

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(toy_transform @ (3.5, 2.5))
print(toy_transform @ (5.5, 4.5))
```

(501750.0, 2998750.0) and (502750.0, 2997750.0). The transform also accepts fractions of a cell, which is how a position *inside* a cell is described.
</details>

## 3. From row and column to x and y, and back

Rasterio has two functions for the everyday conversions:

- `xy(transform, row, col)` gives the **centre** of a cell as (x, y). Add `offset="ul"` for its upper-left corner.
- `rowcol(transform, x, y)` gives the row and column of the cell that **contains** a point.

Watch the order: `xy` and `rowcol` take or give **row before column**, like `array[row, col]`. Only `transform @ (col, row)` takes the column first, because it is the formula with x first.

Going back from x, y to a column and a row needs the **inverse transform**, written `~transform`. It gives fractional positions, such as 3.5 for the middle of column 3. `rowcol` rounds them down to whole cells for you.

**Predict:** which cell contains the point (501750, 2998750)?

In [ ]:
x, y = xy(toy_transform, 2, 3)      # centre of row 2, column 3
print(x, y)

row, col = rowcol(toy_transform, 501750, 2998750)
print(row, col, elev_m[row, col])

In [ ]:
inverse = ~toy_transform
print(inverse @ (501750, 2998750))   # (col, row) as fractions
print(inverse @ (500000, 3000000))   # the top-left corner of the grid

`xy` gave the centre of the summit cell, (501750.0, 2998750.0), and `rowcol` went back to row 2, column 3, where the value is 5,410 m: the top of the made-up mountain.

The inverse transform turned the same point into (3.5, 2.5): halfway across column 3 and halfway down row 2. The grid's top-left corner gives (0.0, 0.0).

### Practice

**Q1 · one line:** Use `xy` to print the centre of the bottom-right cell, at row 4 and column 5.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(xy(toy_transform, 4, 5))
```

(502750.0, 2997750.0), the same as the centre you found with `@ (5.5, 4.5)` in step 2. NumPy may print the numbers as `np.float64(...)`; the values are the same.
</details>

**Q2 · two lines:** A GPS reading (made-up) is at easting 502100 m, northing 2998200 m. Find its row and column with `rowcol`, then print them with the elevation of that cell.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
row, col = rowcol(toy_transform, 502100, 2998200)
print(row, col, elev_m[row, col])
```

Row 3, column 4, elevation 4,980 m. The point is 2,100 m east of the corner (4.2 cells) and 1,800 m south of it (3.6 cells); the cell is the whole-number part of both.
</details>

**Q3 · three lines:** Do the same by hand with the inverse transform: store `col_f, row_f = inverse @ (502100, 2998200)`, print both fractions, then convert them to whole cells with `int()` and print those.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
col_f, row_f = inverse @ (502100, 2998200)
print(col_f, row_f)
print(int(row_f), int(col_f))
```

4.2 and 3.6 (perhaps with tiny rounding digits), then row 3 and column 4, the same cell as `rowcol`. `int()` drops the decimals, which rounds positive numbers down, as in A2.
</details>

## 4. Opening the DEM with rasterio

A **GeoTIFF** is a TIFF image file that also stores where it lies: the transform, the CRS and the nodata value. It is the most common raster file format. Its file name usually ends in `.tif`.

`rasterio` reads and writes GeoTIFFs. Open a file with a `with` block, as you opened text files in Lesson A7:

```python
with rasterio.open(path) as src:
    ...   # use src here
```

`src` is a **dataset**. The file closes itself at the end of the block, even after an error. Inside the block you can ask for its **metadata**, the facts about the raster:

| Attribute | Meaning |
| --- | --- |
| `.profile` | a dictionary of the main facts, ready for writing a new file |
| `.meta` | the same core facts, without the details of how the file is stored |
| `.shape`, `.height`, `.width` | (rows, columns), rows, columns |
| `.count` | the number of bands |
| `.dtypes` | the data type of each band |
| `.nodata` | the value that means "no data" |
| `.crs`, `.transform` | the CRS and the affine transform |
| `.bounds`, `.res` | the box covered, and the cell size (x, y) |

Reading the metadata does not load the values, so it is fast even for very large files.

In [ ]:
DEM_PATH = DATA / "dem_kathmandu_melamchi.tif"

with rasterio.open(DEM_PATH) as src:
    profile = src.profile
    meta = src.meta
    bounds = src.bounds
    res_deg = src.res
    print("shape:", src.shape)
    print("bands:", src.count, "| dtype:", src.dtypes, "| nodata:", src.nodata)
    print("CRS:", src.crs)
    print("bounds:", bounds)
    print("cell size:", res_deg)

print(list(meta))
profile

Read the output line by line:

- **shape** (1620, 2160): 1,620 rows and 2,160 columns.
- **1 band** of type `int16` (whole numbers; step 7 explains data types), with **nodata** −32768. Rasterio always reports nodata as a decimal, `-32768.0`, even for a band of whole numbers.
- **CRS** EPSG:4326: x is longitude and y latitude, in degrees (B4). So the cell size is in **degrees** too.
- **bounds** run from about 85.2° to 85.8° E and from 27.55° to 28.0° N. The edges lie half a cell beyond these round numbers: the source places cell **centres** on whole seconds of arc.
- **cell size** 0.000277…° is 1/3600 of a degree, which is one **arc-second**. This is the "1 arc-second, about 30 m" grid.

`meta` holds eight keys: the driver (file format), dtype, nodata, width, height, count, CRS and transform. The `profile` dictionary repeats these facts and adds how the file is stored: blocks of 256 × 256 cells (`tiled`) and `compress: 'deflate'` (step 7).

### Practice

**Q1 · one line:** `profile` is a normal dictionary (Lesson A4). Print the number of columns it stores under the key `"width"`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(profile["width"])
```

2160 columns. `profile["height"]` gives the 1,620 rows.
</details>

**Q2 · two lines:** Using `bounds`, print how many degrees the DEM spans from west to east (`right - left`) and from south to north (`top - bottom`).

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(bounds.right - bounds.left)
print(bounds.top - bounds.bottom)
```

0.6° from west to east and 0.45° from south to north (with tiny rounding digits). 2,160 cells × 1/3600° = 0.6°, and 1,620 × 1/3600° = 0.45°.
</details>

**Q3 · three lines:** How big is one cell in metres? Use the rule from Lesson B1: one degree of latitude is about 111 km, and one degree of longitude about 111 × cos(latitude) km. Use the middle latitude of the DEM, `(bounds.top + bounds.bottom) / 2`. Print the cell height and width in metres, rounded to 1 decimal place.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
mid_lat_deg = (bounds.top + bounds.bottom) / 2
cell_ns_m = res_deg[1] * 111000
cell_ew_m = res_deg[0] * 111000 * math.cos(math.radians(mid_lat_deg))
print(round(cell_ns_m, 1), round(cell_ew_m, 1))
```

About 30.8 m from north to south but only 27.3 m from west to east. In degrees the cells are square; on the ground they are narrower than they are tall. This matters for plotting (C2) and for slope (C6).
</details>

## 5. Reading the values into a NumPy array

`src.read(1)` reads **band 1** into a 2-D NumPy array: one row of numbers per row of cells. Bands are counted from **1**, not 0, because that is how GeoTIFF numbers them.

`src.read()` with no number reads every band into a **3-D** array of shape (bands, rows, columns). For a one-band DEM that is (1, 1620, 2160).

Once the values are in an array, every tool from Lesson A8 works: `.min()`, `.max()`, `.mean()`, masks, `argmax` and `np.unravel_index`.

Keep `src.transform` too, in a variable: it links every array position to the map.

**Predict:** the facts sheet gives about 1,400 m for Kathmandu. Will the **mean** of the whole DEM be higher or lower than that?

In [ ]:
with rasterio.open(DEM_PATH) as src:
    dem_m = src.read(1)
    dem_transform = src.transform
    all_bands = src.read()

print(type(dem_m), dem_m.shape, dem_m.dtype)
print(all_bands.shape)
print("lowest:", dem_m.min(), "m | highest:", dem_m.max(), "m | mean:", round(dem_m.mean(), 1), "m")

In [ ]:
position = dem_m.argmax()
row, col = np.unravel_index(position, dem_m.shape)
lon, lat = xy(dem_transform, row, col)
print(row, col, dem_m[row, col])
print(round(lon, 4), round(lat, 4))

`dem_m` is a NumPy `ndarray` of 1,620 × 2,160 values, of type `int16`. The DEM runs from 525 m to 3,979 m, with a mean of about 1,531.2 m: higher than Kathmandu's 1,400 m, because the DEM also covers the hills around the valley, which rise to nearly 4,000 m.

The highest cell is at row 31, column 1926: near the northern edge, at about 85.735° E, 27.9914° N. `xy` turned the array position into a longitude and latitude.

### Practice

**Q1 · one line:** What is the **median** elevation of the DEM? Use `np.median` on `dem_m`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
np.median(dem_m)
```

1,435.0 m. Half of the cells are lower and half are higher. The median is lower than the mean (1,531.2 m), because a smaller number of high mountain cells pull the mean up.
</details>

**Q2 · two lines:** What percentage of the cells lie above 2,000 m? Make a mask (A8), then take its mean and multiply by 100.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
above_2000 = dem_m > 2000
print(round(above_2000.mean() * 100, 1))
```

About 15.8% of the cells. The mean of a True/False mask is the share of `True` values, because `True` counts as 1 and `False` as 0.
</details>

**Q3 · three lines:** Find the **lowest** cell: its row, column, elevation, and its longitude and latitude, rounded to 4 decimal places.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
position = dem_m.argmin()
row, col = np.unravel_index(position, dem_m.shape)
lon, lat = xy(dem_transform, row, col)
print(row, col, dem_m[row, col], round(lon, 4), round(lat, 4))
```

Row 361, column 0, 525 m, at 85.2° E and about 27.8997° N: on the western edge of the DEM, in a deep river valley. Column 0 means the lowest ground continues west beyond the clip.
</details>

## 6. The elevation at a place

To read the value at a place, turn its coordinates into a row and a column, then index the array. The dataset can do the first part: `src.index(x, y)` is `rowcol` with the dataset's own transform.

The DEM's CRS is EPSG:4326, so x is the **longitude** and y the **latitude**, in that order, as always since Lesson B1.

Kathmandu Durbar Square lies at about 85.307° E, 27.704° N (approximate). The facts sheet gives about 1,400 m for Kathmandu: a round figure for the whole city. One 30 m cell can differ from it.

The Copernicus DEM measures the top of the surface it sees, which includes buildings and trees. In a city, a cell value can be a roof rather than the ground.

In [ ]:
with rasterio.open(DEM_PATH) as src:
    row, col = src.index(85.307, 27.704)

print(row, col)
print("Durbar Square:", dem_m[row, col], "m")

In [ ]:
hq = gpd.read_file(DATA / "district_headquarters.geojson")
hq_in_dem = hq.cx[bounds.left:bounds.right, bounds.bottom:bounds.top]   # .cx box selection from B5

for name, point in zip(hq_in_dem["name"], hq_in_dem.geometry):
    row, col = rowcol(dem_transform, point.x, point.y)
    print(f"{name:10} row {row:5} col {col:5} {dem_m[row, col]:6,} m")

Durbar Square falls in row 1066, column 385, where the DEM reads **1,300 m**.

Five district headquarters lie inside the DEM. Bhaktapur (1,321 m), Kathmandu (1,308 m) and Lalitpur (1,288 m) sit on the valley floor. Dhulikhel reads 1,522 m, about 200 m higher, and Chautara 1,352 m.

The headquarters points are approximate positions of whole towns. On steep ground, a point a few hundred metres off can read a very different elevation, so treat these values as rough.

`rowcol(dem_transform, ...)` works without opening the file again, because `dem_transform` was saved in step 5. `zip`, from Lesson B7, walks through the names and the points side by side.

### Practice

**Q1 · one line:** What is the elevation at 85.5° E, 27.9° N? Use `rowcol` with `dem_transform` inside the square brackets of `dem_m`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
dem_m[rowcol(dem_transform, 85.5, 27.9)]
```

1,744 m, in the hills north of the valley. `rowcol` gives a (row, col) pair, which NumPy accepts as an index.
</details>

**Q2 · two lines:** Is Everest (86.925° E, 27.9881° N) inside the DEM? Find its row and column, then test whether the column is between 0 and the number of columns, `0 <= col < dem_m.shape[1]`.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
row, col = rowcol(dem_transform, 86.925, 27.9881)
print(row, col, 0 <= col < dem_m.shape[1])
```

Column 6210 is far beyond the last column, 2159, so the answer is `False`. `rowcol` happily computes cells outside the grid; it is up to you to check.
</details>

**Q3 · three lines:** How much higher is Chautara than Kathmandu Durbar Square? Take Chautara's point from `hq_in_dem` (its `name` is `"Chautara"`), read both elevations from `dem_m`, and print the difference in m.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
chautara = hq_in_dem[hq_in_dem["name"] == "Chautara"].geometry.iloc[0]
chautara_m = dem_m[rowcol(dem_transform, chautara.x, chautara.y)]
durbar_m = dem_m[rowcol(dem_transform, 85.307, 27.704)]
print(chautara_m - durbar_m)
```

Chautara's cell reads 1,352 m and Durbar Square's 1,300 m: only 52 m higher. Both values are single 30 m cells at approximate points, so treat the difference as rough.
</details>

## 7. Data types, nodata and bands

Every raster has one **data type** (dtype) for all its cells:

| dtype | Holds | Bytes per cell | Typical use |
| --- | --- | --- | --- |
| `uint8` | whole numbers 0 to 255 | 1 | land-cover classes, photos |
| `int16` | whole numbers −32,768 to 32,767 | 2 | elevations in whole metres |
| `float32` | decimals | 4 | slope, rainfall, results of calculations |

**Nodata** is one chosen value that means "nothing was measured here": outside the survey, under cloud, or a gap in the data. This DEM uses **−32768**, the smallest `int16`, which can never be a real elevation in metres.

This file has **no** nodata cells. Many real files do. If you forget them, −32768 enters your mean like a real elevation. Below, a hole is made **on purpose** to show the damage.

A **band** is one layer of values on the same grid. A DEM has one band; a satellite image has several, one per colour of light (Part D).

In [ ]:
nodata = profile["nodata"]
print(np.iinfo(np.int16))
print("nodata cells in the file:", (dem_m == nodata).sum())

In [ ]:
# Make a hole ON PURPOSE in a copy: 200 x 200 cells set to nodata
holed_m = dem_m.copy()
holed_m[0:200, 0:200] = nodata

print("mean, original:", round(dem_m.mean(), 1))
print("mean, holed, nodata included:", round(holed_m.mean(), 1))

valid = holed_m != nodata
print("mean, holed, nodata left out:", round(holed_m[valid].mean(), 1))

`np.iinfo` shows the range of `int16`: −32,768 to 32,767. The file has 0 nodata cells.

The made-up hole covers only 40,000 of 3.5 million cells (about 1.1%). Yet the mean drops from 1,531.2 m to about 1,143.1 m, because each hole cell counts as −32,768 m. With the hole left out, the mean is about 1,535.2 m, close to the original: only the cells inside the hole are missing.

`.copy()` keeps `dem_m` unchanged; without it, the hole would be cut into the original array. From Lesson C2 on, nodata cells become `NaN` ("not a number", met in B6), which calculations can skip.

### Practice

**Q1 · one line:** Print the dtype of the DEM's only band from the `profile` dictionary.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(profile["dtype"])
```

`int16`: whole metres, 2 bytes per cell.
</details>

**Q2 · two lines:** Compare the memory the array uses (`dem_m.nbytes`) with the size of the file on disk (`DEM_PATH.stat().st_size`, Lesson B3). Print both in MB, rounded to 2 decimal places.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(round(dem_m.nbytes / 1000000, 2))
print(round(DEM_PATH.stat().st_size / 1000000, 2))
```

7.0 MB in memory but only about 2.77 MB on disk. The file is **compressed** (`compress: 'deflate'` in the profile): neighbouring elevations are similar, so they pack well.
</details>

**Q3 · three lines:** Make your own hole on purpose: copy `dem_m` into `gap_m`, set rows 1000 to 1099 and columns 500 to 549 to `nodata`, then print how many cells are nodata and the mean of the valid cells, rounded to 1 decimal place.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
gap_m = dem_m.copy()
gap_m[1000:1100, 500:550] = nodata
print((gap_m == nodata).sum())
print(round(gap_m[gap_m != nodata].mean(), 1))
```

5,000 nodata cells (100 rows × 50 columns), and a mean of about 1,531.5 m for the rest. The hole is small, so the mean barely moves once the hole is left out.
</details>

## 8. A raster of your own, written and read back

To write a GeoTIFF, open a file in mode `"w"` and give rasterio everything a raster needs: `driver="GTiff"`, `height`, `width`, `count` (bands), `dtype`, `crs`, `transform` and `nodata`. Then write the array into band 1 with `dst.write(array, 1)`.

Always **read the file back** and check it. A raster can be stored **upside down**: the values look fine, but north and south are swapped. This happens when the transform's `e` is positive, so that row 0 sits at the bottom.

A classic way to make this mistake is to build the transform by hand, as a move of the origin (`Affine.translation`) times a cell size (`Affine.scale`), with a **positive** y size. Below, the right way with `from_origin` comes first, then the mistake on purpose.

**Predict:** in the wrong transform, will row 0 be north or south of row 4?

In [ ]:
toy_path = OUT / "c1_toy_mountain.tif"

with rasterio.open(
    toy_path, "w", driver="GTiff",
    height=elev_m.shape[0], width=elev_m.shape[1], count=1,
    dtype="int16", crs="EPSG:32645", transform=toy_transform, nodata=-32768,
) as dst:
    dst.write(elev_m.astype("int16"), 1)

with rasterio.open(toy_path) as src:
    back_m = src.read(1)
    print(src.crs, src.bounds)
    print("e is negative:", src.transform.e < 0)
    print("same values:", np.array_equal(back_m, elev_m))
    print("north-west corner cell:", src.index(500100, 2999900))

In [ ]:
# THE OLD BUG, ON PURPOSE: a positive y size turns the grid upside down
wrong_transform = Affine.translation(500000, 3000000) @ Affine.scale(cell_size_m, cell_size_m)
print(wrong_transform.e)
print("row 0 centre y:", xy(wrong_transform, 0, 0)[1])
print("row 4 centre y:", xy(wrong_transform, 4, 0)[1])

The file was written in UTM 45N and read back with the same values. Its bounds run from 2,997,500 m to 3,000,000 m northing, and a point a little inside the north-west corner falls in row 0, column 0: north is at the top. `elev_m.astype("int16")` stored the values as 2-byte whole numbers, matching `dtype="int16"`.

The wrong transform has `e = 500.0`, positive. Row 0 is now centred at y = 3,000,250 m and row 4 at 3,002,250 m: row 0 is **south** of row 4. The grid also lies north of the corner instead of south of it. Nothing warns you; only a check like `transform.e < 0` or a map (C2) reveals it.

### Practice

**Q1 · one line:** How big is the toy file? Print its size in bytes with `.stat().st_size`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(toy_path.stat().st_size)
```

A few hundred bytes (the exact number depends on the library version). Most of it is the header with the transform and CRS; the 30 values take only 60 bytes.
</details>

**Q2 · two lines:** Open the toy file again and print the elevation of the cell that contains the point (501750, 2998750), using `src.index` inside the square brackets.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
with rasterio.open(toy_path) as src:
    print(src.read(1)[src.index(501750, 2998750)])
```

5410: the summit, the same cell you found in step 3. The file kept both the values and their place on the map.
</details>

**Q3 · three lines:** Write the mountain in **feet** as a second file, `c1_toy_mountain_ft.tif`. Convert with 1 m = 3.28084 ft, store the result as `float32`, and use `dtype="float32"`. Then read it back and print the highest value.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
elev_ft = (elev_m * 3.28084).astype("float32")
ft_path = OUT / "c1_toy_mountain_ft.tif"
with rasterio.open(ft_path, "w", driver="GTiff", height=5, width=6, count=1, dtype="float32", crs="EPSG:32645", transform=toy_transform) as dst:
    dst.write(elev_ft, 1)
with rasterio.open(ft_path) as src:
    print(src.read(1).max())
```

About 17,749.34 ft for the 5,410 m summit. Feet have decimals, so the dtype must be a float; written as `int16`, the decimals would be cut off. The `dtype` you declare must match the array you write.
</details>

## 9. When it breaks: reading after the file is closed

You open the DEM, look at its CRS, and then read the values after the `with` block has ended. Run the cell.

In [ ]:
with rasterio.open(DEM_PATH) as src:
    print(src.crs)

dem_again = src.read(1)

The CRS prints, then the last line of the error says:

`RasterioIOError: Dataset is closed: ../data/dem_kathmandu_melamchi.tif`

- **RasterioIOError** is rasterio's own error for input and output problems: reading or writing a file.
- **Dataset is closed** tells you why. The `with` block closed the file when the indented lines ended.

Metadata such as `src.crs` stays available after closing, so the mistake can go unnoticed. Reading **values** needs an open file.

The fix: indent the `read` line so it runs inside the block, and store the result in a variable to use later, as in step 5.

**Try it:** fix the cell.

## Mixed review

These exercises combine everything in this lesson. Each one needs 5 to 10 lines. Plan in words first, then write the code.

### Review 1 · A table of headquarters elevations

Build a pandas table (A9) of the five headquarters in `hq_in_dem`, with the columns `name`, `lon`, `lat`, `row`, `col` and `elevation_m`. Make a list of dictionaries in a loop, turn it into a DataFrame, and sort it from highest to lowest.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
records = []
for name, point in zip(hq_in_dem["name"], hq_in_dem.geometry):
    row, col = rowcol(dem_transform, point.x, point.y)
    records.append({"name": name, "lon": point.x, "lat": point.y,
                    "row": row, "col": col, "elevation_m": dem_m[row, col]})

table = pd.DataFrame(records).sort_values("elevation_m", ascending=False)
table
```

Dhulikhel comes first at 1,522 m, then Chautara (1,352 m), Bhaktapur (1,321 m), Kathmandu (1,308 m) and Lalitpur (1,288 m). This is how you attach raster values to points by hand; Lesson C7 shows faster tools for thousands of points.
</details>

### Review 2 · A short report on the DEM

Print a three-line report with f-strings (A2). Line 1: the number of rows, columns and cells. Line 2: the ground size of the DEM in km, using the cell sizes in metres from step 4 (about 27.3 m × 30.8 m). Line 3: the lowest, highest and mean elevation, and the percentage of cells above 3,000 m.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
mid_lat_deg = (bounds.top + bounds.bottom) / 2
cell_ew_m = res_deg[0] * 111000 * math.cos(math.radians(mid_lat_deg))
cell_ns_m = res_deg[1] * 111000
width_km = dem_m.shape[1] * cell_ew_m / 1000
height_km = dem_m.shape[0] * cell_ns_m / 1000
above_3000_pct = (dem_m > 3000).mean() * 100

print(f"{dem_m.shape[0]:,} rows x {dem_m.shape[1]:,} columns = {dem_m.size:,} cells")
print(f"about {width_km:.1f} km west-east x {height_km:.1f} km south-north, {width_km * height_km:,.0f} km²")
print(f"{dem_m.min()} to {dem_m.max()} m, mean {dem_m.mean():.1f} m, {above_3000_pct:.1f}% above 3,000 m")
```

3,499,200 cells covering about 58.9 km × 50.0 km, roughly 2,943 km². The elevation runs from 525 to 3,979 m, and only about 2.0% of the cells lie above 3,000 m. The area is approximate, because the cell width shrinks slightly from south to north; Lesson C4 measures in UTM metres instead.
</details>

### Review 3 · A west–east profile through Durbar Square

Draw the ground along the row of cells that contains Kathmandu Durbar Square, from the western to the eastern edge of the DEM. Take that row of `dem_m`, get the longitude of every column with `xy(dem_transform, row, cols)`, where `cols = np.arange(dem_m.shape[1])`, and draw a line plot (A10) with labelled axes.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
row, col = rowcol(dem_transform, 85.307, 27.704)
profile_m = dem_m[row, :]
cols = np.arange(dem_m.shape[1])
lons, lats = xy(dem_transform, row, cols)

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(lons, profile_m, color="saddlebrown")
ax.axvline(85.307, color="grey", linestyle="--")
ax.set_xlabel("Longitude (°)")
ax.set_ylabel("Elevation (m)")
ax.set_title(f"West–east profile at {lats[0]:.3f}° N, through Durbar Square (dashed)")
plt.show()
print(profile_m.min(), profile_m.max())
```

As in A10, `fig` is the whole figure and `ax` the plot area inside it. `xy` accepts a whole array of columns and returns arrays of x and y. The flat stretch around the dashed line is the floor of the Kathmandu Valley at about 1,300 m. East of it a ridge rises to 2,030 m, then the ground drops into deep valleys as low as 646 m. A profile is a raster read along a line, a tool engineering geologists use constantly. *Elevation: Copernicus DEM GLO-30 (see the credit at the top).*
</details>

### Review 4 · A tiny raster of the valley box

Lesson B2 used an approximate box around the Kathmandu Valley: 85.2–85.55° E, 27.55–27.82° N. Make a made-up 3 × 4 raster that covers it, with cells of 0.0875° east–west and 0.09° north–south. Fill it with `np.arange(12).reshape(3, 4)` (made-up values), write it as `c1_valley_box.tif` in EPSG:4326, read it back, and print its bounds and the value of the cell that contains Durbar Square.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
values = np.arange(12).reshape(3, 4).astype("int16")
box_transform = from_origin(85.2, 27.82, 0.0875, 0.09)
box_path = OUT / "c1_valley_box.tif"

with rasterio.open(box_path, "w", driver="GTiff", height=3, width=4, count=1,
                   dtype="int16", crs="EPSG:4326", transform=box_transform) as dst:
    dst.write(values, 1)

with rasterio.open(box_path) as src:
    print(src.bounds)
    print(src.read(1)[src.index(85.307, 27.704)])
```

The bounds come back as 85.2 to 85.55° E and 27.55 to 27.82° N, so the transform is right. `.reshape(3, 4)` folds the 12 numbers 0 to 11 into 3 rows of 4. Durbar Square falls in row 1, column 1, whose value is 5: row 1 starts at 4. Four cells × 0.0875° = 0.35° and three × 0.09° = 0.27°.
</details>

## Recap

- A **raster** is a grid of equal **cells** in **rows** and **columns**; each cell holds one value. Row 0 is north and column 0 west. The **cell size** is the resolution.
- The **affine transform** (six numbers `a b c d e f`) places the grid: `transform @ (col, row)` gives map coordinates, `~transform` goes back. Build it with `from_origin`; `e` must be **negative**, or the raster is upside down.
- `xy(transform, row, col)` gives a cell's centre; `rowcol(transform, x, y)` and `src.index(x, y)` give the cell containing a point. In EPSG:4326, x is longitude.
- Open rasters with `with rasterio.open(path) as src:`; read the metadata (`.profile`, `.shape`, `.crs`, `.bounds`, `.res`, `.dtypes`, `.nodata`, `.count`) and the values with `src.read(1)`, inside the block.
- The **dtype** fixes what a cell can hold (`int16` for whole metres). **Nodata** marks missing cells and must be left out of calculations. Write rasters with mode `"w"`, then read them back and check.

### Words you learned

| Word | Meaning |
| --- | --- |
| raster | Data stored as a grid of cells, each with one value |
| cell / pixel | One square of a raster grid |
| cell size / resolution | The ground width of one cell, such as 30 m or 1 arc-second |
| DEM | Digital elevation model: a raster of elevations |
| affine transform | Six numbers that turn a column and row into map x and y |
| `from_origin` | Builds a transform from the top-left corner and the cell sizes |
| inverse transform | `~transform`: turns map x and y back into a fractional column and row |
| `xy` / `rowcol` | Rasterio functions: cell centre from row and column / row and column from x and y |
| GeoTIFF | A TIFF image file that also stores its transform, CRS and nodata value |
| dataset | An open raster file in rasterio, usually called `src` |
| metadata | Facts about a raster: shape, CRS, transform, dtype, nodata, bands |
| profile | Rasterio's dictionary of a raster's main metadata |
| arc-second | 1/3600 of a degree; about 30.8 m of latitude |
| band | One layer of values on a raster grid |
| dtype | The data type of every cell, such as `int16` or `float32` |
| nodata | A chosen value that marks cells with no measurement |
| compression | Storing a file in fewer bytes without losing values |
| elevation profile | The ground drawn along a line, as elevation against distance |

## Next

**C2 · Plotting rasters:** draw the DEM as a real map, with the right colours, a colourbar in metres, true proportions, and the district boundaries and headquarters on top.

<!-- pygile-nav -->
**PyGILE Course** · lesson 25 of 39 · [Course home](../README.md)  
← Previous: [B12 · Interactive web maps: Nepal you can zoom](../B_Vector/B12_Interactive_web_maps.ipynb) · Next: [C2 · Plotting rasters: an honest map of the Kathmandu Valley DEM](C02_Plotting_rasters.ipynb) →